In [1]:
!pip install llama-index
!pip install llama-index-llms-groq
!pip install llama-index-embeddings-huggingface

  Using cached llama_index_llms_openai-0.7.10-py3-none-any.whl.metadata (3.0 kB)
Using cached llama_index_llms_openai-0.7.10-py3-none-any.whl (28 kB)
  Attempting uninstall: llama-index-llms-openai
    Found existing installation: llama-index-llms-openai 0.8.3
    Uninstalling llama-index-llms-openai-0.8.3:
      Successfully uninstalled llama-index-llms-openai-0.8.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-openai-like 0.8.1 requires llama-index-llms-openai<0.9,>=0.8.0, but you have llama-index-llms-openai 0.7.10 which is incompatible.


In [46]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.llms.groq import Groq
from getpass import getpass
from llama_index.core.tools import QueryEngineTool, FunctionTool
from llama_index.core.agent import ReActAgent
from llama_index.core.agent.workflow import FunctionAgent


In [56]:
groq_api_key = "gsk_..."

In [60]:
llm= Groq(model="openai/gpt-oss-120b", api_key=groq_api_key)
Settings.llm=llm

In [7]:
documents= SimpleDirectoryReader(input_files= ["Salma_Ibrahim_Ali_Resume.txt"]).load_data()
documents

[Document(id_='07cba547-ea8f-44e5-bfc5-9b32af808757', embedding=None, metadata={'file_path': 'Salma_Ibrahim_Ali_Resume.txt', 'file_name': 'Salma_Ibrahim_Ali_Resume.txt', 'file_type': 'text/plain', 'file_size': 3765, 'creation_date': '2026-10-24', 'last_modified_date': '2026-10-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Neural Radiance Fields (NeRF) represent a cutting-edge technique in 3D computer graphics that uses neural networks to generate novel views of complex 3D scenes. Unlike traditional graphics pipelines that rely on explicit geometry and texture maps, NeRF models learn an implicit volumetric representation of a scene 

In [8]:
index= VectorStoreIndex.from_documents(documents,embed_model="local:sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [61]:
query_engine= index.as_query_engine(llm=llm)

In [62]:
#to test rag system first
response =query_engine.query("what does a neural radiance represent?")
print (response)

A neural radiance field encodes, for any point in space and any viewing direction, the amount of light that would be emitted (its color) together with how much that point blocks or scatters light (its volume density). In other words, it is an implicit, learned description of a scene’s appearance and opacity that can be integrated along camera rays to produce photorealistic images from new viewpoints.


In [24]:
file_tool = QueryEngineTool.from_defaults(
    query_engine,
    name="neural_radiance",
    description="Use this tool to answer questions about Neural Radiance Fields (NeRF), "
        "including volumetric rendering, positional encoding, training, dynamic scenes, "
        "and the typical 'neural radiance number'. "
        "The input should be a complete natural-language question. "
        "Do not use it for math calculations or unrelated topics."
)

In [37]:
def add (a:float, b:float)-> float:
  """ Add two numbers and returns the sum"""
  return a + b
add_tool = FunctionTool.from_defaults(
    fn=add,
    name="add",
    description="Adds two numbers. ALWAYS use this tool for any addition, never calculate manually."
)

In [50]:
agent = FunctionAgent(
    tools=[add_tool, file_tool],
    llm=llm,
    verbose=True,
    system_prompt=(
        "You are a tool-using assistant. "
        "For ANY fact about neural radiance, call the neural_radiance tool first. "
        "For ANY addition, even a simple one, you MUST call the add_tool. "
        "Never compute or guess numbers yourself."
)
)

In [30]:
response = await agent.run(user_msg="what does a neural radiance represent?")
print(str(response))

[tick] add: AgentWorkflowStartEvent(user_msg='what does a neural radiance represent?', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='what does a neural radiance represent?')])], current_agent_name='Agent')
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='what does a neural radiance represent?')])], current_agent_name='Agent')
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='think

In [51]:
response = await agent.run(user_msg="what is the sum of the number o neural radiance plus 32432584 ?")
print(str(response))

[tick] add: AgentWorkflowStartEvent(user_msg='what is the sum of the number o neural radiance pl...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='what is the sum of the number o neural radiance plus 32432584 ?')])],...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a tool-using assistant. For ANY fact about neural radianc...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaT